# KNN — Original/Masked Color hiperparametre optimizasyonu

**Ortak protokol:** 10.015 HAM10000 görüntüsü, 439 renk özelliği; aynı stratified train/validation/test indeksleri (`random_state=42`). Eğitim üzerinde 3-fold `RandomizedSearchCV` ve **Macro F1** optimizasyonu. Validation yalnızca model seçimi ve karşılaştırma için; **test kullanılmaz**. Original ve Masked ayrı ayrı aranır. `StandardScaler` gerektiğinde CV içinde `Pipeline` kullanılır.

**Dikkat:** Masked sonuçlar gerçek maske kullandığından, gerçek kullanımda otomatik segmentasyon gerekir. Aynı lezyona ait tekrar görüntüler bulunuyorsa bu image-level split veri sızıntısı yaratabilir; metadata ile grup bazlı bölmeyi ayrıca ele almak gerekir.

## 1 — Yollar, özellikler ve ortak veri bölmesi

In [8]:
from pathlib import Path
import sys, json, time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from joblib import dump

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from src.config import BASE_DIR
from src.data_loader import load_dataset
FEATURE_DIR = BASE_DIR / "outputs" / "features"
RESULT_DIR = BASE_DIR / "outputs" / "models" / "color_optimized"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

X_sources = {
    "original": FEATURE_DIR / "X_original_color_300x225.npy",
    "masked": FEATURE_DIR / "X_masked_color_300x225.npy",
}
y = np.load(FEATURE_DIR / "y_labels_color_300x225.npy", allow_pickle=False)
ids = np.load(FEATURE_DIR / "image_ids_color_300x225.npy", allow_pickle=False)
df = load_dataset(require_masks=True).reset_index(drop=True)
assert np.array_equal(ids.astype(str), df["image"].to_numpy(dtype=str)), "Görüntü ID sıralaması uyuşmuyor"
assert np.array_equal(y.astype(str), df["label"].to_numpy(dtype=str)), "Etiket sıralaması uyuşmuyor"
for variant, path in X_sources.items():
    if not path.exists(): raise FileNotFoundError(f"Önce color features notebook'unu çalıştır: {path}")
    tmp = np.load(path, mmap_mode="r", allow_pickle=False)
    assert tmp.shape == (len(y), 439), (variant, tmp.shape)
    assert np.issubdtype(tmp.dtype, np.number)

idx = np.arange(len(y))
dev_idx, test_idx = train_test_split(idx, test_size=0.15, random_state=42, stratify=y)
train_idx, val_idx = train_test_split(dev_idx, test_size=0.15/0.85, random_state=42, stratify=y[dev_idx])
assert len(set(train_idx)&set(val_idx)) == len(set(train_idx)&set(test_idx)) == len(set(val_idx)&set(test_idx)) == 0
classes = np.unique(y)
print('Train:',len(train_idx),'Validation:',len(val_idx),'Test (dokunulmayacak):',len(test_idx))
print('Çıktı:',RESULT_DIR)

Train: 7009 Validation: 1503 Test (dokunulmayacak): 1503
Çıktı: C:\Users\Deniz TOZAN\OneDrive - Analythinx Bilişim Hizmetleri Ticaret Anonim Şirketi\WORK\Calismalar\uygulama\Kodlar\Health\outputs\models\color_optimized


## 2 — Model ve hiperparametre uzayı

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

MODEL_NAME = 'KNN'
base_model = Pipeline([("scale", StandardScaler()), ("model", KNeighborsClassifier(n_jobs=2))])
param_space = {"model__n_neighbors": [3, 5, 7, 11, 15, 21, 31, 45, 65], "model__weights": ["uniform", "distance"], "model__p": [1, 2], "model__leaf_size": [20, 30, 50]}
N_ITER = 20
CV_FOLDS = 3
SEARCH_JOBS = 2  # RAM / işlemci kullanımına göre 1 yapabilirsiniz
print(base_model)
print(param_space)

Pipeline(steps=[('scale', StandardScaler()),
                ('model', KNeighborsClassifier(n_jobs=2))])
{'model__n_neighbors': [3, 5, 7, 11, 15, 21, 31, 45, 65], 'model__weights': ['uniform', 'distance'], 'model__p': [1, 2], 'model__leaf_size': [20, 30, 50]}


## 3 — Original ve Masked için ayrı optimizasyon ve validation
İlk çalıştırmada her iki model aranır. Önceden kaydedilmiş çıktıları tekrar üretmek istiyorsanız hücreyi yeniden çalıştırın. Sonuçlar önceki aynı adlı dosyaların üzerine yazılır.

In [10]:
print("Model:", base_model)
print("Parametreler:", param_space)
print("Etiket tipi:", y.dtype)
print("Sınıflar:", np.unique(y))

Model: Pipeline(steps=[('scale', StandardScaler()),
                ('model', KNeighborsClassifier(n_jobs=2))])
Parametreler: {'model__n_neighbors': [3, 5, 7, 11, 15, 21, 31, 45, 65], 'model__weights': ['uniform', 'distance'], 'model__p': [1, 2], 'model__leaf_size': [20, 30, 50]}
Etiket tipi: <U10
Sınıflar: ['AKIEC' 'BCC' 'BKL' 'DF' 'MEL' 'NV' 'VASC']


In [11]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Sınıflar:", label_encoder.classes_)
print("Kodlanmış sınıflar:", np.unique(y_encoded))

Sınıflar: ['AKIEC' 'BCC' 'BKL' 'DF' 'MEL' 'NV' 'VASC']
Kodlanmış sınıflar: [0 1 2 3 4 5 6]


In [15]:
print("X_train tipi:", type(X_train))
print("X_train shape:", np.shape(X_train))
print("y_train shape:", y[train_idx].shape)

X_train tipi: <class 'numpy.ndarray'>
X_train shape: (7009, 439)
y_train shape: (7009,)


In [19]:
rows=[]
cv=StratifiedKFold(n_splits=CV_FOLDS,shuffle=True,random_state=42)
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

print("Sınıf eşleştirmeleri:")
for i, name in enumerate(encoder.classes_):
    print(i, "=", name)


for variant, path in X_sources.items():
    t0=time.time()
    X=np.load(path, mmap_mode="r", allow_pickle=False)
    X_train=np.asarray(X[train_idx], dtype=np.float32)
    X_val=np.asarray(X[val_idx], dtype=np.float32)
    if not np.isfinite(X_train).all() or not np.isfinite(X_val).all():
        raise ValueError(f"{variant}: NaN/Inf var")
    print(f"\n=== {MODEL_NAME}: {variant} ===",flush=True)
    search=RandomizedSearchCV(
        estimator=base_model,param_distributions=param_space,n_iter=N_ITER,
        scoring="f1_macro",cv=cv,refit=True,n_jobs=SEARCH_JOBS,
        random_state=42,verbose=1,return_train_score=False,error_score="raise"
    )
    #search.fit(X_train,y[train_idx])

    search.fit(X_train, y_encoded[train_idx])
    pred_encoded = search.best_estimator_.predict(X_val)
    train_pred_encoded = search.best_estimator_.predict(X_train)

    pred = encoder.inverse_transform(pred_encoded)
    train_pred = encoder.inverse_transform(train_pred_encoded)
    r={
        "Model":MODEL_NAME,"Variant":variant,"CV_Macro_F1":float(search.best_score_),
        "Train_Macro_F1":float(f1_score(y[train_idx],train_pred,average="macro",zero_division=0)),
        "Val_Accuracy":float(accuracy_score(y[val_idx],pred)),
        "Val_Macro_F1":float(f1_score(y[val_idx],pred,average="macro",zero_division=0)),
        "Val_Macro_Precision":float(precision_score(y[val_idx],pred,average="macro",zero_division=0)),
        "Val_Macro_Recall":float(recall_score(y[val_idx],pred,average="macro",zero_division=0)),
        "MEL_Recall":float(recall_score(y[val_idx],pred,labels=["MEL"],average="macro",zero_division=0)),
        "MEL_Precision":float(precision_score(y[val_idx],pred,labels=["MEL"],average="macro",zero_division=0)),
        "Predicted_Classes":int(len(np.unique(pred))),"Best_Params":json.dumps(search.best_params_,ensure_ascii=False),
        "Search_Seconds":round(time.time()-t0,1),"Train_N":len(train_idx),"Val_N":len(val_idx),"Test_N":len(test_idx),
    }
    rows.append(r)
    prefix=RESULT_DIR/f"{MODEL_NAME}_{variant}"
    pd.DataFrame(search.cv_results_).to_csv(str(prefix)+"_cv_results.csv",index=False)
    pd.DataFrame({"image_id":ids[val_idx],"actual":y[val_idx],"predicted":pred}).to_csv(str(prefix)+"_val_predictions.csv",index=False)
    pd.DataFrame(confusion_matrix(y[val_idx],pred,labels=classes),index=classes,columns=classes).to_csv(str(prefix)+"_confusion_matrix.csv")
    dump(search.best_estimator_,str(prefix)+"_model.joblib")
    print('En iyi CV Macro F1:',round(search.best_score_,4),'Validation Macro F1:',round(r['Val_Macro_F1'],4))
    print('Parametreler:',search.best_params_)
    print(classification_report(y[val_idx],pred,labels=classes,zero_division=0))
    del X_train,X_val,search
summary=pd.DataFrame(rows).sort_values("Val_Macro_F1",ascending=False)
summary.to_csv(RESULT_DIR/f"{MODEL_NAME}_summary.csv",index=False)
display(summary)
print('Test seti kullanılmadı.')

Sınıf eşleştirmeleri:
0 = AKIEC
1 = BCC
2 = BKL
3 = DF
4 = MEL
5 = NV
6 = VASC

=== KNN: original ===


Fitting 3 folds for each of 20 candidates, totalling 60 fits
En iyi CV Macro F1: 0.3369 Validation Macro F1: 0.3734
Parametreler: {'model__weights': 'distance', 'model__p': 1, 'model__n_neighbors': 5, 'model__leaf_size': 50}
              precision    recall  f1-score   support

       AKIEC       0.32      0.18      0.23        49
         BCC       0.33      0.22      0.26        77
         BKL       0.49      0.38      0.43       165
          DF       0.57      0.22      0.32        18
         MEL       0.49      0.28      0.35       167
          NV       0.79      0.93      0.85      1006
        VASC       0.50      0.10      0.16        21

    accuracy                           0.72      1503
   macro avg       0.50      0.33      0.37      1503
weighted avg       0.68      0.72      0.69      1503


=== KNN: masked ===
Fitting 3 folds for each of 20 candidates, totalling 60 fits
En iyi CV Macro F1: 0.3795 Validation Macro F1: 0.3916
Parametreler: {'model__weights': 'distanc

,Model,Variant,CV_Macro_F1,Train_Macro_F1,Val_Accuracy,Val_Macro_F1,Val_Macro_Precision,Val_Macro_Recall,MEL_Recall,MEL_Precision,Predicted_Classes,Best_Params,Search_Seconds,Train_N,Val_N,Test_N
1,KNN,masked,0.379460,1.0,0.730539,0.391581,0.479117,0.364201,0.221557,0.506849,7,"{""model__weights"": ""distance"", ""model__p"": 1, ...",26.0,7009,1503,1503
0,KNN,original,0.336888,1.0,0.717898,0.373433,0.498716,0.330227,0.275449,0.494624,7,"{""model__weights"": ""distance"", ""model__p"": 1, ...",27.0,7009,1503,1503


Test seti kullanılmadı.


## 4 — Çıktılar
`outputs/models/color_optimized/` altında model başına **özet CSV**, her varyant için **CV sonuçları, validation tahminleri, confusion matrix ve `.joblib` model** kaydedilir. Önce dört model notebook’unu çalıştırın; ardından değerlendirme notebook’unu açın.

In [ ]:
import winsound

winsound.Beep(300, 3000)
print("Ses testi tamamlandı.")